# Creating the CREDIT layer

CREDIT: Custom Residential Demand Indicative Target

--------------------------------------------------------------------------

**Original code:** [Alexandros Korkovelos](https://github.com/akorkovelos) <br />
**Conceptualization & Methodological review :**  [Alexandros Korkovelos](https://github.com/akorkovelos) <br />
**Updates, Modifications:** [Alexandros Korkovelos](https://github.com/akorkovelos) <br />
**Adapted for any country (DRC example):** Bachirou <br />
**Funding:** The World Bank

--------------------------------------------------------------------------
Predicting electricity demand in currently un-electrified settlement is a challenge. Many energy access efforts use the multi-tier framework (MTF) as a point of reference; OnSSET in particular, uses this approach as it provides a good basis of (investment) comparison between different levels of access. Access tiers are assigned to settlements based on selection criteria; the most common approach (also adopted in GEP) is to assign tier-targets to urban/rural settlements respectively. This approach allows for limited spatial allocation of targets let alone it does not take into account newly available spatial information.

The **Custom Residential Electricity Demand Indicative Target** (CREDIT) has been developed so as to provide a spatially explicit, alternative approach of predicting electricity demand targets in currently un-electrified settlements, by using poverty and GDP maps. The process is described step-by-step below. It works for **any country**: set the country and parameters in the *Settings* cell and run all cells. The values shipped in this notebook are for the **Democratic Republic of the Congo (DRC)**. This notebook has been developed to support replication of the process at any stage.

## Settings — change these for another country

Everything country-specific is in the cell below. To run the model for another country:

1. Set `COUNTRY_ISO3`, `COUNTRY_NAME` and `YEAR`.
2. Check the poverty table column names (`POVERTY_CODE_COL`, `POVERTY_NAME_COL`, `POVERTY_VALUE_COL`) after Step 1.1.a.
3. Poverty values are joined to the admin boundaries automatically: by PCode when the official HDX
   boundaries exist, otherwise by name (exact, then fuzzy). If Step 1.1.a still reports unmatched
   units, add them to `NAME_ALIASES`.
4. Replace `MTF_KWH_PER_CAPITA` with demand targets suited to the country (household size).
5. Leave `PROJECTED_CRS = None` to use the country's UTM zone automatically, or set an EPSG code.

In [ ]:
# ---------------- Country ----------------
COUNTRY_ISO3 = "COD"                                  # ISO3 code (geoBoundaries + Kummu GDP)
COUNTRY_NAME = "Congo, Democratic Republic of the"   # name used to find the HDX/OPHI poverty dataset
COUNTRY_LABEL = "DRC"                                 # short label for titles and file names
YEAR = 2024                                           # GDP year (Kummu v4: 1990, 1995, ..., 2020, 2024)
KEEP_GLOBAL_GDP = True                                # False = delete the 1.4 GB global GDP raster after
                                                      # clipping (re-downloaded if another country/year is run)

# ---------------- Poverty data ----------------
ADMIN_LEVEL = "ADM1"                    # admin level of the poverty table (OPHI MPI is ADM1)
ADMIN_SOURCE = "auto"                   # "auto" = HDX COD-AB (joins by PCode), else geoBoundaries;
                                        # or force "hdx" / "geoboundaries"
POVERTY_CODE_COL = "Admin 1 PCode"      # admin-unit code column in the poverty table
POVERTY_NAME_COL = "Admin 1 Name"       # admin-unit name column in the poverty table
POVERTY_VALUE_COL = "Headcount Ratio"   # poverty headcount column (0-100 %)
NAME_ALIASES = {}                       # optional {boundary name: poverty-table name} overrides

# ---------------- Model parameters ----------------
PROJECTED_CRS = None                    # e.g. "EPSG:32734"; None = UTM zone of the country centroid
RESOLUTION_M = 1000                     # output pixel size in metres
POVERTY_BINS = [20, 40, 60, 80, 100]    # equal-interval bins for poverty headcount (%)
GDP_CLASSES = 5                         # number of natural-breaks classes for GDP
WEIGHT_POV, WEIGHT_GDP = 0.5, 0.5       # weights of the two layers (must sum to 1)
RANDOM_SEED = 0                         # makes the GDP natural-breaks classes reproducible

# MTF demand targets (kWh/capita/year) for index values 0, 1, 2, 3, 4, 5.
# DRC: Malawi MTF targets scaled by household size (4.6 / 5.3 ≈ 0.868), Tier 1 ≈ 7.6, Tier 5 ≈ 590.
# NOTE: verify against official national MTF calibration data before production use.
MTF_INDEX = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
MTF_KWH_PER_CAPITA = [1, 7.6, 153.3, 299.0, 444.7, 590.4]

## Preparing the environment

Before starting with the code it is important that the following modules are imported. All helper functions live in `credit_utils.py`. 

In [ ]:
import os
import sys
import logging

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import mapclassify as mc
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap
import earthpy.plot as ep
from shapely.geometry import Point, mapping
from osgeo import gdal

ROOT_DIR = os.path.abspath(os.curdir)
sys.path.insert(0, ROOT_DIR)
import credit_utils as cu

%matplotlib inline

# ---------------- Folders and file names ----------------
tag = COUNTRY_LABEL                      # prefix for country-specific files
poverty_root = os.path.join(ROOT_DIR, "poverty_data")
admin_path = os.path.join(ROOT_DIR, "admin")
gdp_path = os.path.join(ROOT_DIR, "GDP_data")
temp_path = os.path.join(ROOT_DIR, "tmp", COUNTRY_ISO3)                # one folder per country,
sample_output = os.path.join(ROOT_DIR, "sample_output", COUNTRY_ISO3)  # so countries never overwrite each other
for p in [poverty_root, admin_path, gdp_path, temp_path, sample_output,
          os.path.join(poverty_root, "tabular"), os.path.join(poverty_root, "shapefile"),
          os.path.join(poverty_root, "raster")]:
    os.makedirs(p, exist_ok=True)

paths = {
    "poverty_csv":   os.path.join(poverty_root, "tabular", f"{COUNTRY_ISO3.lower()}_poverty_data.csv"),
    "poverty_shp":   os.path.join(poverty_root, "shapefile", f"{tag}_admin1.shp"),
    "poverty_tif":   os.path.join(poverty_root, "raster", f"{tag}_poverty_map.tif"),
    "poverty_tif_wgs84": os.path.join(poverty_root, "raster", f"{tag}_poverty_map_in_CRS.tif"),
    "gdp_tif":       os.path.join(gdp_path, f"GDP_{YEAR}_{COUNTRY_ISO3.lower()}_clipped.tif"),
    "poverty_pts":   os.path.join(temp_path, f"poverty_map_{COUNTRY_ISO3.lower()}_in_CRS.shp"),
    "gdp_pts":       os.path.join(temp_path, f"GDP_{YEAR}_{COUNTRY_ISO3.lower()}_clipped.shp"),
    "poverty_class": os.path.join(temp_path, "poverty_map_in_CRS_classified.tif"),
    "gdp_class":     os.path.join(temp_path, f"GDP_{YEAR}_{COUNTRY_ISO3.lower()}_clipped_classified.tif"),
    "ahp_option1":   os.path.join(sample_output, "AHP_option_1.tif"),
    "ahp_clipped":   os.path.join(sample_output, "clipped_sum_.tif"),
    "credit_tif":    os.path.join(sample_output, "CREDIT.tif"),
}

print(f"GDAL version: {gdal.__version__}")
print(f"Active environment: {sys.prefix}")
print(f"Country: {COUNTRY_NAME} ({COUNTRY_ISO3}), year {YEAR}")
print(f"Results folder: {sample_output}")

## Step 1. Collection of raw GIS data

First, we collect available data for the area/country of interest. In this case we will use the national [GDP, PPP (purchasing-power-parity) values for 2015 in constant 2011 U.S. dollars (USD)](https://gep-onsset.readthedocs.io/en/latest/GIS%20data%20collection.html#gdp-ppp). Poverty maps can be acquired from the [WorldPop](https://www.worldpop.org/geodata/listing?id=23) database in raster format. However, their availability is limited to only a few countries. Therefore here, we describe how can one use tabular data available on administrative level. Values used, indicate the share of population living below the poverty line (defined as $2/day). 

### 1.1 Importing poverty data

You may receive the poverty data in different ways; you may refer to:

- **1.1.a** in case you receive data in csv format
- **1.1.b** in case you receive data in shp
- **1.1.c** in case poverty data is already in .tiff format

#### 1.1.a Poverty data available in tabular (csv) format

**Download the poverty table (OPHI MPI from HDX)** — skipped if already on disk

In [ ]:
poverty_data = cu.extract_country_tabular(COUNTRY_NAME, out_dir=poverty_root)

# Save a cleaned copy (HXL tag row removed) for reference
if not os.path.exists(paths["poverty_csv"]):
    poverty_data.to_csv(paths["poverty_csv"], index=False)
    print(f"[OK] Saved cleaned poverty data to {paths['poverty_csv']}")
else:
    print(f"[SKIP] {paths['poverty_csv']} already exists, not overwriting.")

poverty_data.head(3)

**Import administrative layer as vector file at the level the poverty data is available**

The tabular data above do not have a spatial index. In order to appoint them to the relative administrative unit, the latter needs to be imported.

In [ ]:
# HDX COD-AB boundaries (same PCodes as the poverty table) or geoBoundaries as fallback
admin_gdf, admin_info = cu.load_admin_boundary(COUNTRY_ISO3, ADMIN_LEVEL, admin_path, source=ADMIN_SOURCE)
ADMIN_NAME_COL = admin_info["name_col"]

# Projected CRS used for all metric operations (rasterizing at RESOLUTION_M)
if PROJECTED_CRS is None:
    PROJECTED_CRS = cu.utm_crs_for(admin_gdf)
print(f"Projected CRS: {PROJECTED_CRS}")
admin_gdf.head()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
admin_gdf.plot(ax=ax, edgecolor='brown', alpha=0.2)
ax.set_aspect('equal', 'box')

**Merge tabular and admininstrative data**

In [ ]:
pov_per_admin = cu.merge_poverty_to_admin(
    admin_gdf, poverty_data,
    poverty_value_col=POVERTY_VALUE_COL,
    admin_name_col=ADMIN_NAME_COL,
    poverty_name_col=POVERTY_NAME_COL,
    admin_code_col=admin_info["code_col"],
    poverty_code_col=POVERTY_CODE_COL,
    name_aliases=NAME_ALIASES,
)
pov_per_admin[[ADMIN_NAME_COL, POVERTY_NAME_COL, 'pov_hcr', 'match_method']]

In [ ]:
fig, ax = plt.subplots(figsize=(15, 15))
pov_per_admin.sort_values('pov_hcr', ascending=True).plot(ax=ax, column='pov_hcr', legend=True, alpha=0.5)
ax.set_aspect('equal', 'box')
txt = ax.set_title(f'Distribution of poverty (%) per admin unit in {COUNTRY_LABEL} ({ADMIN_LEVEL})')

---
**NOTE**

The dataset above is not projected. In case you want to spatially process the layer make sure you project it to the appropriate coordinate system.

---

**Export geodataframe as shapefile if needed**

In [ ]:
pov_per_admin.to_file(paths["poverty_shp"])
pov_per_admin_proj = pov_per_admin.to_crs(PROJECTED_CRS)

---
**NOTE!!**

No matter if you create the geodataframe from the .csv data or you import it directly from a shp file, the next step requires rasterization; that is, converting the vector polygon to a raster (.tiff) output.

---

**Rasterize the poverty geo-dataframe**

In [ ]:
cu.rasterize_vector(pov_per_admin_proj, paths["poverty_tif"], field="pov_hcr", res=RESOLUTION_M)

with rasterio.open(paths["poverty_tif"]) as src:
    src_extent = np.asarray(src.bounds)[[0, 2, 1, 3]]
    plt.figure(figsize=(15, 15))
    plt.imshow(src.read(1), cmap='Reds', extent=src_extent)
    plt.show()

You may want to re-project and save the raster to the original CRS for future use and/or dissemination

In [ ]:
# Back to WGS84 (same CRS as the GDP raster); factor=1 keeps the pixel count
cu.reproj(paths["poverty_tif"], paths["poverty_tif_wgs84"], new_crs="epsg:4326", factor=1)

### 1.2 Importing GDP data

Here we assume that GDP data is retrieved in raster format (see [source](https://gep-onsset.readthedocs.io/en/latest/GIS%20data%20collection.html#gdp-ppp)). You may clip the raster using the admin mask for the area of interest, as follows.

In [ ]:
# If the country GDP raster already exists, it is reused and nothing is downloaded.
# Otherwise the Kummu v4 global raster (1.4 GB) is downloaded once and the country is clipped out.
da_gdp = cu.get_country_gdp(COUNTRY_ISO3, YEAR, gdp_path=gdp_path, out_path=paths["gdp_tif"],
                            keep_global=KEEP_GLOBAL_GDP)
print("GDP sum (PPP):", float(da_gdp.sum()))

Plotting

In [ ]:
# GDP PPP with quintile breaks
with rasterio.open(paths["gdp_tif"]) as gdb_src:
    gdp = gdb_src.read(1).astype(float)
    gdp[gdp == gdb_src.nodata] = np.nan

classes = 5
valid = gdp[np.isfinite(gdp)]
breaks = np.nanpercentile(valid, np.linspace(0, 100, classes + 1))
norm = colors.BoundaryNorm(breaks, classes)
cmap = ListedColormap(["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#225ea8"])

fig, ax = plt.subplots(figsize=(15, 15))
chm_plot = ax.imshow(gdp, cmap=cmap, norm=norm)
ax.set_title(f"GDP PPP ({YEAR}) in {COUNTRY_LABEL} — quintile breaks")
cbar = ep.colorbar(chm_plot)
boundary_means = [np.mean([norm.boundaries[ii], norm.boundaries[ii-1]]) for ii in range(1, len(norm.boundaries))]
cbar.set_ticks(boundary_means)
cbar.set_ticklabels(["Very Low", "Low", "Medium", "High", "Very High"])
ax.set_axis_off()
plt.show()

## Step 2. Re-classify raster layers

The second step on the process requires that both - poverty and GDP - maps are re-classified using a 1-5 scale as shown in the table below. 

| Initial   poverty layer | Classification| Initial GDP layer | Classification |
|-------------------------|---------------|-------------------|----------------|
| 0 ≤   pov. rate < 0.2   | 5             | I1 ≤ GDP < I3     | 1              |
| 0.2 ≤   pov. rate < 0.4 | 4             | I2 ≤ GDP < I3     | 2              |
| 0.4 ≤   pov. rate < 0.6 | 3             | I3 ≤ GDP < I4     | 3              |
| 0.6 ≤   pov. rate < 0.8 | 2             | I4 ≤ GDP < I5     | 4              |
| pov. rate ≥   0.8       | 1             | GDP ≥ I5          | 5              |

The poverty map is classified using the equal interval method. The GDP map using natural breaks mentod. The classification method can have an impact on the final product layer. Please make sure you understand the differences between the methods and select the one that suits best in your case. 

You may read about this on:

- [GISing with python](https://automating-gis-processes.github.io/CSC18/lessons/L3/reclassify.html)
- [Pysal git](https://github.com/pysal/mapclassify) & [Pysal doc](https://pysal.org/scipy2019-intermediate-gds/stochastic/gds4-visualization.html)
- [GIS wiki](http://wiki.gis.com/wiki/index.php/Classification)
- [ArcGIS support](https://pro.arcgis.com/en/pro-app/help/mapping/layer-properties/data-classification-methods.htm)

---
**NOTE!!**

In the relevant [publication](https://www.mdpi.com/1996-1073/12/7/1395) GDP is reclassifed using geometric interval classification in Arcgis. You can replicate the process if you have access to ArcMap. In this notebook we will make use of the pysal library to retrieve the classification bins using different methods and then reclassify the rasters based on those bins.

---

In [ ]:
# Country outline (ADM0) from the admin layer, used to clip rasters
shapes = [mapping(admin_gdf.dissolve().geometry.iloc[0])]

### Calculate classification bins

Perhaps there are faster, more sophisticated or pythonic ways to do this but here we work as follows.

- Convert raster files to geo-dataframes
- Get the bins using pysal for the selected classification method

Convert poverty map to shp to geo-dataframe & re-classify

In [ ]:
cu.raster_to_point_shp(paths["poverty_tif_wgs84"], paths["poverty_pts"], band_no=1)
poverty_gdf = gpd.read_file(paths["poverty_pts"])

poverty_bin = mc.UserDefined(poverty_gdf.VALUE, bins=POVERTY_BINS)
poverty_bin

In [ ]:
# Lower poverty -> higher class: <20 % -> 5, 20-40 -> 4, ..., >=80 -> 1
cu.reclassify_raster(paths["poverty_tif_wgs84"], paths["poverty_class"],
                     bins=poverty_bin.bins, lookup=[(0, 3)])

Convert gdp map to shp to geo-dataframe & re-classify

In [ ]:
cu.raster_to_point_shp(paths["gdp_tif"], paths["gdp_pts"], band_no=1)
gdp_gdf = gpd.read_file(paths["gdp_pts"])

np.random.seed(RANDOM_SEED)   # NaturalBreaks uses random starts; the seed makes runs identical
gdp_bin = mc.NaturalBreaks(gdp_gdf.VALUE, k=GDP_CLASSES)
gdp_bin

In [ ]:
# Higher GDP -> higher class; NaN (outside the country) -> 0 / nodata
cu.reclassify_raster(paths["gdp_tif"], paths["gdp_class"], bins=gdp_bin.bins,
                     lookup=[(0, 5), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1)], mask_nan=True)

## Step 3. Sum the re-classified layers 

Here we sum the (weighted) re-classified layers to create a combined index ranging from 1-5. The operation is relatively simple and can be represented as follows:

<center>(poverty_reclas X weight_1) + (gdp_reclas X weight_2)   |   where weight_1 + weight_2 = 1</center>

---
**Note!**

- One easy way of doing this is by using raster calculator tools in a GIS environment ([Qgis](https://docs.qgis.org/2.8/en/docs/user_manual/working_with_raster/raster_calculator.html) or [ArcMap](https://desktop.arcgis.com/en/arcmap/10.3/tools/spatial-analyst-toolbox/raster-calculator.htm)).


- In case one wants to work in python only, alternative options include the [gdal_calc](https://gdal.org/programs/gdal_calc.html) code or even [rio calc](https://rasterio.readthedocs.io/en/latest/topics/calc.html). However, this requires that the raster layers are entirely alligned, as the operation is taking place on their respective arrays (see **Option 1** below)


- Alternatively, one might follow a less efficient approach that involves several transformations. Nevertheless, the approach works and yields the desired result regardless of the size of input rasters (see **Option 2** below).

Feel free to use use as per your understanding and convenience!

---

### Option 1 -- Re-sizing raster arrays based on shape

In [ ]:
cu.combine_rasters(paths["poverty_class"], paths["gdp_class"], paths["ahp_option1"],
                   weight_pov=WEIGHT_POV, weight_gdp=WEIGHT_GDP)
cu.clip_raster(paths["ahp_option1"], paths["ahp_clipped"], shapes)

---

**Note!**
In case the process reports arraey out of bounds simply re-assign raster1 and raster 2.


---

In [ ]:
with rasterio.open(paths["ahp_clipped"]) as src:
    sample_out = src.read(1)

cmap = ListedColormap(["white", "tan", "springgreen", "darkgreen"])
classes = 5
breaks = (np.arange(classes) / float(classes)) * sample_out.max()   # for visualization only
norm = colors.BoundaryNorm(list(breaks), classes)

fig, ax = plt.subplots(figsize=(15, 15))
chm_plot = ax.imshow(sample_out, cmap=cmap, norm=norm)
ax.set_title(f"AHP layer — {COUNTRY_LABEL}")
cbar = ep.colorbar(chm_plot)
boundary_means = [np.mean([norm.boundaries[ii], norm.boundaries[ii - 1]]) for ii in range(1, len(norm.boundaries))]
cbar.set_ticks(boundary_means)
cbar.set_ticklabels(['Low', 'Medium', 'High', 'Very High'])
ax.set_axis_off()
plt.show()

### Option 2 -- Converting raster to geodataframe and work accordingly

### 3.1 Create a "base" point layer as a csv file

Here we use one of the geo-dataframes (either poverty or gdp) to create a vector point layer that we will use as base. It is suggested to use the one that yield the resolution that best suits each case. Here we proceed with the GDP geo-dataframe.

In [ ]:
# Base point layer = GDP pixels (from Step 2), with an index and X/Y coordinates
base = gdp_gdf.copy()
base['idx'] = range(1, len(base) + 1)
base['lon'] = base.geometry.x
base['lat'] = base.geometry.y
base.drop(columns='geometry').to_csv(os.path.join(temp_path, "base.csv"), index=False)
base.head(3)

### 3.2 Extract raster values to points defined by the "base" csv file

In [ ]:
# Sample both classified rasters at every base point
values = cu.extract_raster_values_to_points(base, {
    "gdp_class": paths["gdp_class"],
    "poverty_class": paths["poverty_class"],
})

pov_gdp = pd.concat([base[['idx', 'lon', 'lat']], values], axis=1)

# Points outside a raster get the middle class (3)
for col in ["gdp_class", "poverty_class"]:
    pov_gdp[col] = pov_gdp[col].fillna(3).astype(int)

pov_gdp.to_csv(os.path.join(temp_path, "pov_gdp_points.csv"), index=False)
pov_gdp.head(2)

### 3.4 Adding the two classified columns using weighting factors

In [ ]:
pov_gdp['AHP_index'] = (pov_gdp['gdp_class'] * WEIGHT_GDP) + (pov_gdp['poverty_class'] * WEIGHT_POV)

pov_gdp_gdf = gpd.GeoDataFrame(pov_gdp, geometry=[Point(xy) for xy in zip(pov_gdp['lon'], pov_gdp['lat'])],
                               crs='epsg:4326')

fig, ax = plt.subplots(figsize=(25, 25))
pov_gdp_gdf.sort_values('AHP_index', ascending=True).plot(ax=ax, column='AHP_index', legend=True,
                                                         markersize=3, cmap="viridis", alpha=0.5)
ax.set_aspect('equal', 'box')
ax.set_title(f'Result of the AHP process — {COUNTRY_LABEL}')
plt.savefig(os.path.join(sample_output, 'AHP_result_option2.png'), bbox_inches='tight')

## Step 4. Interpolate and create the CREDIT layer

The combined index (0-5) is converted to a residential demand target (kWh/capita/year) by linear interpolation between the MTF tier targets set in the *Settings* cell.

In [ ]:
pov_gdp_gdf["CREDIT"] = cu.credit_from_index(pov_gdp_gdf["AHP_index"], MTF_INDEX, MTF_KWH_PER_CAPITA)

fig, ax = plt.subplots(figsize=(25, 25))
pov_gdp_gdf.sort_values('CREDIT', ascending=True).plot(ax=ax, column='CREDIT', legend=True,
                                                      markersize=3, cmap="viridis", alpha=0.5)
ax.set_aspect('equal', 'box')
ax.set_title(f'CREDIT layer (kWh/capita/year) — {COUNTRY_LABEL}')
plt.savefig(os.path.join(sample_output, 'Sample_result.png'), bbox_inches='tight')
logging.getLogger('matplotlib.font_manager').disabled = True

In [ ]:
# Rasterize the CREDIT points in the projected CRS and save
pov_gdp_gdf_proj = pov_gdp_gdf.to_crs(PROJECTED_CRS)
cu.rasterize_vector(pov_gdp_gdf_proj, paths["credit_tif"], field="CREDIT", res=RESOLUTION_M)
print("Saved:", paths["credit_tif"])